# PW7 - Encoding, System Design and Debugging

**Team:** Noé Berdoz, Michael Strefeler

## Our starting level

- **Noé Berdoz**: Bachelor in Business Information Technology (Informatique de gestion) at HE-Arc, completed in 2026. This programme focuses on software development and business processes and includes little mathematics, so _TODO: what was new for Noé in this PW_. He had almost no experience in machine learning before this course.
- **Michael Strefeler**: Bachelor in Computer and Communication Systems (ISC) at HEIG-VD, completed in 2025, with a specialisation in data science, so he had already seen some machine learning before this course. _TODO: what was new for Michael in this PW_

## Time spent

| Exercise | Time |
|---|---|
| Ex. 1 - Data preparation (UBS use case) | _TODO_ |
| Ex. 2 - Debugging drugs dataset | _TODO_ |
| Ex. 3 - Linear regression optimisation | _TODO_ |
| **Total** | **_TODO_** |

## Bonus request

We would like to apply for the bonus for this PW.

**Why.** _TODO: in our own words, why we should get the bonus, measured against our starting levels_

**What we did as bonus**

- **Ex. 1** _TODO_
- **Ex. 2** _TODO_
- **Ex. 3** _TODO_

# Exercise 1 - Analysis

The bank UBS is offering the possibility to invest money in investment funds. A fund is composed of financial values such as stocks or bonds. For example, a fund composed mostly of stocks has more return potential but is more risky in case of stock market recession. There are thousands of funds available, see https://fundgate.ubs.com/. The probability to invest or not in a fund is conditioned by the profile of the fund and of the client. For example, a younger client with no child is potentially more interested into funds composed with stocks, showing higher risks but also higher potential returns. A family father will be more inclined to invest into low-risk funds. UBS want to build a system as illustrated on Figure~\ref{fig:ubs_system}, taking as input a set of values characterizing the fund and the client profile.

An investment fund can be characterized by the following elements: 

- The name of the fund.
- The current value of 1 share in the fund, expressed in CHF.
- The proportion of stock and bonds composing the fund (2 values in percentage).
- A vector of float values with the 5 last yearly returns over years from 2015 to 2019 (5 values expressed in percentage).
- A level of risk expressed with A, B, C, D, E with A representing the highest risk and E representing the lowest risk level.
- A sectorial information such as technology, pharmaceutical, financial. There are 24 different sectors available in UBS funds.
-  As the set of funds are worldwide, the emiting location is also available with the address of the managing entity of the fund, e.g. Market Street 1234, New York, USA.

A client profile contains the following information: 

- First name and last name of the client.
- The mother tongue of the client (mostly de, fr, it and en but other languages are present).
- The age of the client.
- The number of children of the client.
- The current wealth of the client that could be used to buy funds, expressed in CHF (total of cash available in the different accounts, not yet invested in funds).
- The postal code of the address of the client.
- A level of acceptance to risk expressed with A, B, C, D, E with A representing the highest level of acceptance of risk and E representing the lowest acceptance of risk.

Answer the following questions:

1. For each available information in the fund and client profile, explain how you would prepare the data: encoding, normalization, outlier treatment, etc.
2. How could you collect targets (output of the system) to train the system? How would you prepare the different sets?

**Be as comprehensive as possible.**  Don't limit your explanation to the "how" but also the "why".

---

**For each available information in the fund and client profile, explain why and how you would prepare the data: encoding, normalization, outlier treatment, etc.**

### Fund Profile Preprocessing

#### Fund Name
We drop the fund name from the model inputs. As a unique identifier, one-hot encoding it across thousands of funds would force the model to memorize instances rather than generalize, while leaving any new fund without valid inputs. We retain the name strictly as a database key to map predictions back to their targets.

#### Share value
We apply a log transformation followed by a z-score normalization (Log -> Z-Norm), using the mean and standard deviation computed only on the training set. Share values typically follow a long tailed distribution (where most funds are moderately priced while a few reach extreme values) which warrants log scaling. Because high nominal share price does not imply fund performance or quality, we will use the validation set to test whether keeping this feature actually improves predictive performance.

#### Stocks & Bonds Allocation
Both values are kept and divided by 100, which corresponds to a min-max rescaling over the known bounds \[0, 100\]. Crucially, we do not drop one in favor of the other as funds frequently hold cash or other assets, meaning the bond allocation cannot simply be deduced as $100 - \text{Stocks}$.

#### Yearly Returns (5 year window)
We retain the five yearly returns as individual inputs ordered chronologically from $t - 5$ to $t - 1$ relative to the decision date, forming a sliding window through time. Each position is independently z-normalized using its column's training statistics. A log transform is ruled out here since returns can be negative, whereas z-normalization gracefully accomodates moderate outliers such as an exceptional year. For younger funds lacking a full five-year history, our strategy is to impute missing years with the training mean of that column, which naturally maps to 0 post normalization.

#### Risk level
We map the categorical risk rating to an ordered scale: $A = 1.0$, $B = 0.75$, $C = 0.5$, $D = 0.25$, and $E = 0.0$. This maintains the ordinal hierarchy between grades while naturally constraining values to the \[0, 1 \] interval without further rescaling. A one-hot encoding would fail here by treating adjacent risk tiers as though they were as distant as opposite extremes.

#### Sector
We encode the sector using a one-hot vector across 24 categories. Because sectors have no intrinsic hierarchy, integer encoding (1 to 24) would impose an artificial metric distance between industries (e.g., arbitrarily placing technology closer to pharmaceuticals than to finance). For funds operating across several industries, the vector can naturally hold fractional weights reflecting portfolio allocation (such as 0.6 and 0.4) rather than a strict binary flag.

#### Managing Entity Address
We geocode the address into latitude and longitude while stripping away street names and numbers. Because funds are allocated globally, relying on raw longitude introduces an artificial discontinuity at the antimeridian ($\pm 180^\circ$ near Hawaii). To avoid this artifact, we convert the coordinates into 3D Cartesian coordinates on the unit sphere:
$x = \cos(\text{lat})\cos(\text{lon})$, $y = \cos(\text{lat})\sin(\text{lon})$, and $z = \sin(\text{lat})$

This eliminates spatial boundaries and natively bounds all input values between -1 and 1.

### Client Profile Preprocessing

#### Client Name
We drop both the first and last name from the model inputs. Names act purely as identifiers and have no causal relationship with investment choices. One-hot encoding thousands of unique names would merely cause the model to memorize individual clients. Furthermore, names could leak demographic proxies such as gender or ethnic background, creating an ethical risk of biased predictions. We retain only an internal client ID as a database key to join features with their ground-truth labels.

#### Mother Tongue
We encode mother tongue as a one-hot vector across five inputs: German (de), French (fr), Italian (it), English (en), and an aggregated category for all other languages. Because language categories are nominal with no natural hierarchy, integer encoding is inappropriate. Grouping rare languages avoids extreme sparsity and allows the model to process previously unseen languages at test time without failing. Since language can still serve as a regional or demographic proxy, we maintain strict validation checks by keeping this feature if it demonstrates measurable predictive value on the validation set, and we audit model outputs to ensure fairness across linguistic groups.

#### Age
We retain age as a single numerical feature rescaled via min-max normalization, using the minimum and maximum ages observed exclusively on the training set. Age exhibits natural physiological bounds and no severe outliers, making it an ideal candidate for min-max scaling. If an unseen client in production is older than the oldest training client, the resulting scaled value will slightly exceed 1.0, which standard linear layers and neural architectures handle without issue.

#### Number of Children
We keep the number of children as a single numeric feature because order and magnitude are informative (family obligations typically correlate with lower risk tolerance). We apply a z-score normalization using the training mean and standard deviation. Large families represent rare but legitimate data points (the classic "normal outliers" for which z-normalization is recommended). As an alternative candidate, we also test clipping at 4 children followed by a min-max rescaling to \[0, 1\], treating this choice as an empirical hyperparameter to compare on the validation set.

#### Available Wealth
We apply a logarithmic transform followed by a z-score normalization ($\log(1 + x) \rightarrow \text{Z-norm}$), fitted only on the training set. Wealth follows a pronounced long-tailed distribution where the vast majority of clients hold modest savings while a few hold millions. Standard min-max scaling would compress almost all clients into a narrow band near zero. We use $\log(1 + x)$ rather than $\log(x)$ to keep the transformation well-defined if a client holds zero wealth. Following the course guidance on treating normalization schemes as hyperparameters, our second candidate is percentile clipping followed by min-max scaling, evaluated and selected via validation performance.

#### Postal Code
We replace the raw postal code with the geographic coordinates (latitude and longitude) of the associated municipality, yielding two distinct inputs rescaled via min-max normalization based on training extremes. Postal codes are discrete administrative labels rather than continuous quantities. For example, 4001 Basel is numerically closer to 1003 Lausanne than to 8001 Zürich, which is completely misrepresenting their true spatial geography. Because Switzerland is geographically compact and situated well away from the antimeridian (+-180°), latitude and longitude contain no discontinuities in this case, rendering a planar min-max scaling sufficient without requiring 3D spherical projection. One-hot encoding the postal codes is discarded as it would inflate dimensionality and fail on unseen zip codes.

#### Risk Acceptance
We encode client risk acceptance on the exact same ordered numerical scale as the fund risk rating: $A = 1.0$, $B = 0.75$, $C = 0.5$, $D = 0.25$, and $E = 0.0$. Aligning both features onto an identical metric space allows the model to compute direct comparisons, distances, and interactions between a client's risk appetite and a fund's actual volatility profile.

### Global Preprocessing Rule and Feature Count
All preprocessing parameters (mean $\mu$, standard deviation $\sigma$, minimum and maximum) are computed strictly on the training set and applied unchanged to the validation, test, and live production sets to prevent data leakage.

In total each fund yields 36 input features (1 share value, 2 asset allocations, 5 historical returns, 1 risk level, 24 sector flags, and 3 spatial coordinates), while each client yields 11 input features (5 language flags, 1 age, 1 family size, 1 wealth value, 2 coordinates, and 1 risk score). Concatenating both vectors provides the classifier with exactly 47 numerical inputs per fund-client pair.

**How could you collect targets (output of the system) to train the system? How would you prepare the different sets?**

TODO: answer